In [1]:
from openai import OpenAI
import pandas as pd
import minsearch


## Ingestion

In [2]:
df = pd.read_csv('../data/cleaned_data.csv')
df.head()

,id,task,category,difficulty,duration_estimate,framework_name,reasoning,instructions,tags
0,1,Write a 2-page project summary,work,medium,45,Time Blocking,Time Blocking helps allocate a clear writing w...,"Block 45 minutes, outline key points, write su...",work;writing;planning
1,2,Organize pantry shelves,home,medium,40,Task Decomposition,Breaking the pantry into sections makes the ta...,"Empty one shelf, clean it, categorize items, r...",home;organization
2,3,Practice 20-minute mobility routine,fitness,easy,20,Daily Rituals,Mobility benefits from consistent daily habits.,"Warm up joints, perform hip circles, shoulder ...",fitness;mobility;routine
3,4,Review study notes for chapter 3,study,easy,25,Spaced Repetition,Reviewing notes at intervals improves retention.,"Read notes, highlight key ideas, summarize in ...",study;memory
4,5,Write a blog outline,creative,medium,30,Mind Mapping,Mind Mapping helps structure ideas visually.,"Create central topic, branch subtopics, add su...",creative;writing


In [3]:
documents = []

for _, row in df.iterrows():
    documents.append({
        "id": str(row["id"]),
        "task": row["task"],
        "category": row["category"],
        "difficulty": row["difficulty"],
        "duration_estimate": str(row["duration_estimate"]),
        "framework_name": row["framework_name"],
        "reasoning": row["reasoning"],
        "instructions": row["instructions"],
        "tags": row["tags"],
    })



In [4]:
index = minsearch.Index(['task', 'category', 'difficulty',
       'framework_name', 'reasoning', 'instructions', 'tags'],
        keyword_fields=["id", 'duration_estimate'])


In [5]:
index.fit(documents)

print(documents[0])

{'id': '1', 'task': 'Write a 2-page project summary', 'category': 'work', 'difficulty': 'medium', 'duration_estimate': '45', 'framework_name': 'Time Blocking', 'reasoning': 'Time Blocking helps allocate a clear writing window.', 'instructions': 'Block 45 minutes, outline key points, write summary, revise.', 'tags': 'work;writing;planning'}


In [6]:
q = "How can I organize my workspace efficiently using a structured method?"

## RAG flow

In [7]:
client = OpenAI()

def search(query):
    boost = {}

    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=10
    )

    return results

In [8]:
index.search(q, num_results=10)

[{'id': '185',
  'task': 'Clean bedroom nightstand',
  'category': 'home',
  'difficulty': 'easy',
  'duration_estimate': '15',
  'framework_name': 'Task Batching',
  'reasoning': 'Batching cleaning tasks increases efficiency.',
  'instructions': 'Wipe surface, organize items, clean drawer.',
  'tags': 'home;cleaning'},
 {'id': '191',
  'task': 'Organize digital notes',
  'category': 'home',
  'difficulty': 'medium',
  'duration_estimate': '35',
  'framework_name': 'GTD',
  'reasoning': 'GTD helps categorize digital clutter.',
  'instructions': 'Sort notes, create folders, delete old items.',
  'tags': 'home;digital'},
 {'id': '110',
  'task': 'Clean laundry area',
  'category': 'home',
  'difficulty': 'medium',
  'duration_estimate': '25',
  'framework_name': 'Task Batching',
  'reasoning': 'Batching cleaning tasks increases efficiency.',
  'instructions': 'Wipe surfaces, organize detergents, clean floor.',
  'tags': 'home;cleaning'},
 {'id': '210',
  'task': 'Clean living room shelve

In [9]:


prompt_template = """
You're a productivity advisor. Answer the QUESTION based on the CONTEXT from our productivity tasks dataset.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT:
{context}
""".strip()

entry_template = """
task: {task}
category: {category}
difficulty: {difficulty}
duration_estimate: {duration_estimate}
instructions: {instructions}
reasoning: {reasoning}
tags: {tags}
""".strip()

def build_prompt(query, search_results):
    context = ""

    for doc in search_results:
        context += entry_template.format(**doc) + "\n\n"

    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt



In [10]:
search_results = search(q)
prompt = build_prompt(q, search_results)


In [11]:

def llm(prompt, model='gpt-4o-mini'):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content


In [12]:
def rag(query, model='gpt-4o-mini'):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, model=model)
    return answer

In [13]:
question = "How can I organize my workspace efficiently using a structured method?"
answer = rag(question)
print(answer)

To organize your workspace efficiently using a structured method, consider the following steps based on proven strategies:

1. **Clear the Workspace**: Start by clearing your desk or workspace of all items. This will give you a fresh perspective on what you have and need.

2. **Sort Items**: Go through your items and sort them into categories based on their function or type. This aligns with the Getting Things Done (GTD) method, which helps categorize and clarify clutter.

3. **Use Containers and Labels**: For each category, use containers to store similar items together and label them. This will help you find what you need quickly and maintain organization.

4. **Prioritize Essentials**: Place only the essential items that you use frequently on your desk while storing away the rest. This keeps your workspace decluttered and focused.

5. **Break Tasks into Sections**: If your workspace requires more extensive organization, break it down into smaller sections. Focus on one section at a 

In [14]:
query = "What key points should I include in my outline for the project summary?"
ans = rag(query)
print(ans)



For your project summary outline, you should include the following key points:

1. **Introduction**
   - Brief overview of the project and its objectives.

2. **Achievements**
   - List of significant accomplishments or milestones reached during the project.

3. **Proposed Next Steps**
   - Outline future actions or phases of the project following the current summary.

4. **Challenges Encountered**
   - A brief mention of any issues faced during the project.

5. **Solutions Implemented**
   - Overview of solutions that were applied to address the challenges.

6. **Deadlines**
   - Set timelines related to future actions or ongoing tasks for clarity.

This structure will ensure clarity and coherence in your summary while following a SMART framework for effective communication.


## Retrieval evaluation

In [15]:
df_question = pd.read_csv('../data/ground-truth-retrieval.csv')
df_question.head()

,id,question
0,1,What key points should I include in my outline...
1,1,How can I effectively manage my time during th...
2,1,What strategies can I use to ensure my summary...
3,1,How should I approach the revision process aft...
4,1,Are there any specific writing techniques that...


In [16]:
ground_truth = df_question.to_dict(orient='records')

In [17]:

ground_truth[0]

{'id': 1,
 'question': 'What key points should I include in my outline for the project summary?'}

In [18]:
def hit_rate(relevance_total):
    cnt = 0

    for line in relevance_total:
        if True in line:
            cnt = cnt + 1

    return cnt / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:
                total_score = total_score + 1 / (rank + 1)

    return total_score / len(relevance_total)

In [19]:
def minsearch_search(query):
    boost = {}

    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=10
    )

    return results

In [20]:
from tqdm.auto import tqdm

In [21]:
def evaluate(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        doc_id = str(q['id'])
        results = search_function(q)
        relevance = [str(d['id']) == doc_id for d in results]
        relevance_total.append(relevance)

    return {
        'hit_rate': hit_rate(relevance_total),
        'mrr': mrr(relevance_total),
    }


In [22]:
print(ground_truth[0])

{'id': 1, 'question': 'What key points should I include in my outline for the project summary?'}


In [23]:
print(minsearch_search(ground_truth[0]["question"]))

[{'id': '1', 'task': 'Write a 2-page project summary', 'category': 'work', 'difficulty': 'medium', 'duration_estimate': '45', 'framework_name': 'Time Blocking', 'reasoning': 'Time Blocking helps allocate a clear writing window.', 'instructions': 'Block 45 minutes, outline key points, write summary, revise.', 'tags': 'work;writing;planning'}, {'id': '217', 'task': 'Write project summary draft', 'category': 'work', 'difficulty': 'medium', 'duration_estimate': '30', 'framework_name': 'SMART Goals', 'reasoning': 'SMART structure improves clarity in summaries.', 'instructions': 'Write intro, list achievements, propose next steps.', 'tags': 'work;writing'}, {'id': '242', 'task': 'Write project improvement summary', 'category': 'work', 'difficulty': 'medium', 'duration_estimate': '30', 'framework_name': 'SMART Goals', 'reasoning': 'SMART structure improves clarity in improvement summaries.', 'instructions': 'List issues, propose solutions, set deadlines.', 'tags': 'work;analysis'}, {'id': '19

In [24]:

evaluate(ground_truth, lambda q: minsearch_search(q['question']))

  0%|          | 0/1250 [00:00<?, ?it/s]

{'hit_rate': 0.8072, 'mrr': 0.6059936507936513}

## Finding the best parameters

In [25]:
df_validation = df_question[:100]
df_testing = df_question[100:]

In [26]:
import random

def simple_optimize(param_ranges, objective_function, n_iterations=10):
    best_params = None
    best_score = float('-inf')  # Assuming we're minimizing. Use float('-inf') if maximizing.

    for _ in range(n_iterations):
        # Generate random parameters
        current_params = {}
        for param, (min_val, max_val) in param_ranges.items():
            if isinstance(min_val, int) and isinstance(max_val, int):
                current_params[param] = random.randint(min_val, max_val)
            else:
                current_params[param] = random.uniform(min_val, max_val)
        
        # Evaluate the objective function
        current_score = objective_function(current_params)
        
        # Update best if current is better
        if current_score > best_score:  # Change to > if maximizing
            best_score = current_score
            best_params = current_params
    
    return best_params, best_score

In [27]:
gt_val = df_validation.to_dict(orient='records')

In [28]:
import random

def simple_optimize(param_ranges, objective_function, n_iterations=10):
    best_params = None
    best_score = float('-inf')  # Assuming we're minimizing. Use float('-inf') if maximizing.

    for _ in range(n_iterations):
        # Generate random parameters
        current_params = {}
        for param, (min_val, max_val) in param_ranges.items():
            if isinstance(min_val, int) and isinstance(max_val, int):
                current_params[param] = random.randint(min_val, max_val)
            else:
                current_params[param] = random.uniform(min_val, max_val)
        
        # Evaluate the objective function
        current_score = objective_function(current_params)
        
        # Update best if current is better
        if current_score > best_score:  # Change to > if maximizing
            best_score = current_score
            best_params = current_params
    
    return best_params, best_score

In [29]:

def minsearch_search(query, boost=None):
    if boost is None:
        boost = {}

    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=10
    )

    return results

param_ranges = {
    "task": (0.0, 3.0),
    "instructions": (0.0, 3.0),
    "reasoning": (0.0, 3.0),
    "tags": (0.0, 3.0),
    "category": (0.0, 3.0),
    "difficulty": (0.0, 3.0),
}

def objective(boost_params):
    def search_function(q):
        return minsearch_search(q['question'], boost_params)

    results = evaluate(gt_val, search_function)
    return results['mrr']


In [30]:
simple_optimize(param_ranges, objective, n_iterations=20)

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

...

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

({'task': 1.4335265718979693,
  'instructions': 1.9034180820049942,
  'reasoning': 0.08137676245487957,
  'tags': 1.5598097285804697,
  'category': 1.5714825957072693,
  'difficulty': 0.7702530103725097},
 0.6957301587301591)

In [31]:
def minsearch_improved(query):
    boost = {
        'exercise_name': 2.11,
        'type_of_activity': 1.46,
        'type_of_equipment': 0.65,
        'body_part': 2.65,
        'type': 1.31,
        'muscle_groups_activated': 2.54,
        'instructions': 0.74
    }

    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=10
    )

    return results

evaluate(ground_truth, lambda q: minsearch_improved(q['question']))

  0%|          | 0/1250 [00:00<?, ?it/s]

{'hit_rate': 0.7912, 'mrr': 0.5851914285714294}

## RAG evaluation

In [32]:
prompt2_template = """
You are an expert evaluator for a RAG system.
Your task is to analyze the relevance of the generated answer to the given question.
Based on the relevance of the generated answer, you will classify it
as "NON_RELEVANT", "PARTLY_RELEVANT", or "RELEVANT".

Here is the data for evaluation:

Question: {question}
Generated Answer: {answer_llm}

Please analyze the content and context of the generated answer in relation to the question
and provide your evaluation in parsable JSON without using code blocks:

{{
  "Relevance": "NON_RELEVANT" | "PARTLY_RELEVANT" | "RELEVANT",
  "Explanation": "[Provide a brief explanation for your evaluation]"
}}
""".strip()

In [33]:
len(ground_truth)

1250

In [34]:
record = ground_truth[0]
question = record['question']

In [35]:
answer_llm = rag(question) 
print(answer_llm)


In your outline for the project summary, you should include the following key points:

1. **Introduction**:
   - Brief overview of the project, including its purpose and objectives.

2. **Achievements**:
   - List of accomplishments and milestones reached during the project.

3. **Proposed Next Steps**:
   - Suggestions for future actions or projects based on the current outcomes.

4. **Issues Identified**:
   - Discussion of any challenges or problems encountered during the project.

5. **Proposed Solutions**:
   - Recommendations for addressing the identified issues.

6. **Deadlines**:
   - Suggested timelines for the implementation of the proposed next steps and solutions.

This structured approach will enhance the clarity and effectiveness of your project summary.


In [36]:
prompt = prompt2_template.format(question=question, answer_llm=answer_llm)
print(prompt)

You are an expert evaluator for a RAG system.
Your task is to analyze the relevance of the generated answer to the given question.
Based on the relevance of the generated answer, you will classify it
as "NON_RELEVANT", "PARTLY_RELEVANT", or "RELEVANT".

Here is the data for evaluation:

Question: What key points should I include in my outline for the project summary?
Generated Answer: In your outline for the project summary, you should include the following key points:

1. **Introduction**:
   - Brief overview of the project, including its purpose and objectives.

2. **Achievements**:
   - List of accomplishments and milestones reached during the project.

3. **Proposed Next Steps**:
   - Suggestions for future actions or projects based on the current outcomes.

4. **Issues Identified**:
   - Discussion of any challenges or problems encountered during the project.

5. **Proposed Solutions**:
   - Recommendations for addressing the identified issues.

6. **Deadlines**:
   - Suggested ti

In [37]:
llm(prompt)

'{\n  "Relevance": "RELEVANT",\n  "Explanation": "The generated answer provides a comprehensive list of key points to include in an outline for a project summary, directly addressing the question. Each point is relevant to summarizing a project effectively, covering essential aspects such as introduction, achievements, proposed next steps, identified issues, proposed solutions, and deadlines."\n}'

In [38]:
import json
df_sample = df_question.sample(n=200, random_state=1)
sample = df_sample.to_dict(orient='records')

In [39]:
evaluations = []

for record in tqdm(sample):
    question = record['question']
    answer_llm = rag(question) 

    prompt = prompt2_template.format(
        question=question,
        answer_llm=answer_llm
    )

    evaluation = llm(prompt)
    evaluation = json.loads(evaluation)

    evaluations.append((record, answer_llm, evaluation))

  0%|          | 0/200 [00:00<?, ?it/s]

In [40]:
df_eval = pd.DataFrame(evaluations, columns=['record', 'answer', 'evaluation'])

df_eval['id'] = df_eval.record.apply(lambda d: d['id'])
df_eval['question'] = df_eval.record.apply(lambda d: d['question'])


df_eval['relevance'] = df_eval.evaluation.apply(lambda d: d['Relevance'])
df_eval['explanation'] = df_eval.evaluation.apply(lambda d: d['Explanation'])

del df_eval['record']
del df_eval['evaluation']

In [41]:
df_eval.relevance.value_counts(normalize=True)

relevance
RELEVANT           0.765
PARTLY_RELEVANT    0.170
NON_RELEVANT       0.065
Name: proportion, dtype: float64

In [42]:
df_eval.to_csv('../data/rag-eval-gpt-4o-mini.csv', index=False)

In [43]:
evaluations_gpt4o = []

for record in tqdm(sample):
    question = record['question']
    answer_llm = rag(question, model='gpt-4o') 

    prompt = prompt2_template.format(
        question=question,
        answer_llm=answer_llm
    )

    evaluation = llm(prompt)
    evaluation = json.loads(evaluation)
    
    evaluations_gpt4o.append((record, answer_llm, evaluation))

  0%|          | 0/200 [00:00<?, ?it/s]

In [44]:
df_eval = pd.DataFrame(evaluations_gpt4o, columns=['record', 'answer', 'evaluation'])

df_eval['id'] = df_eval.record.apply(lambda d: d['id'])
df_eval['question'] = df_eval.record.apply(lambda d: d['question'])

df_eval['relevance'] = df_eval.evaluation.apply(lambda d: d['Relevance'])
df_eval['explanation'] = df_eval.evaluation.apply(lambda d: d['Explanation'])

del df_eval['record']
del df_eval['evaluation']

In [45]:
df_eval.relevance.value_counts()

relevance
RELEVANT           156
PARTLY_RELEVANT     36
NON_RELEVANT         8
Name: count, dtype: int64

In [46]:
df_eval.relevance.value_counts(normalize=True)

relevance
RELEVANT           0.78
PARTLY_RELEVANT    0.18
NON_RELEVANT       0.04
Name: proportion, dtype: float64

In [47]:
df_eval.to_csv('../data/rag-eval-gpt-4o.csv', index=False)

In [48]:
model="gpt-5.4-mini"
llm(prompt, model=model)

'{\n  "Relevance": "PARTLY_RELEVANT",\n  "Explanation": "The answer does provide a one-sentence gratitude example, which is relevant to the question. However, it adds unnecessary context about journaling tasks and does not directly focus on the simplest way to express gratitude in one sentence, making it only partially aligned."\n}'

In [49]:
evaluations_gpt_54mini = []

for record in tqdm(sample):
    question = record['question']
    answer_llm = rag(question, model= model) 

    prompt = prompt2_template.format(
        question=question,
        answer_llm=answer_llm
    )

    evaluation = llm(prompt, model='gpt-5.4-mini')
    evaluation = json.loads(evaluation)
    
    evaluations_gpt_54mini.append((record, answer_llm, evaluation))

  0%|          | 0/200 [00:00<?, ?it/s]

In [50]:
df_eval = pd.DataFrame(evaluations_gpt4o, columns=['record', 'answer', 'evaluation'])

df_eval['id'] = df_eval.record.apply(lambda d: d['id'])
df_eval['question'] = df_eval.record.apply(lambda d: d['question'])

df_eval['relevance'] = df_eval.evaluation.apply(lambda d: d['Relevance'])
df_eval['explanation'] = df_eval.evaluation.apply(lambda d: d['Explanation'])

del df_eval['record']
del df_eval['evaluation']

In [51]:
df_eval.relevance.value_counts()

relevance
RELEVANT           156
PARTLY_RELEVANT     36
NON_RELEVANT         8
Name: count, dtype: int64

In [52]:
df_eval.relevance.value_counts(normalize=True)

relevance
RELEVANT           0.78
PARTLY_RELEVANT    0.18
NON_RELEVANT       0.04
Name: proportion, dtype: float64